# Build Standard_GT
Run cells in order with the project Python kernel (Python 3.10+).
Install `requirements-gt.txt` and `ipykernel` first. This notebook needs no GPU.

This profile matches the supplied MATLAB stitcher. Read `docs/gt_convention.md`.
Images are copied unchanged; CSV poses use full supplied precision.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'build_standard_gt.py').exists(), 'Open inside the project'
sys.path.insert(0, str(ROOT))
from build_standard_gt import build_standard_gt

DATASET = ROOT / 'dataset'
POSE_CSV = DATASET / 'positions_angles_camera1.csv'
OUTPUT = ROOT / 'Standard_GT'
SEQUENCE_ID = 'sequence001'  # Use a new name for another export.
MAP_NAME = None  # Set the actual CARLA map name if known.
CARLA_VERSION = None


## Export
Put original `equi_*.png` images and the headerless source CSV in `dataset/`.
The CSV columns are x,y,z,pitch,yaw,roll; filename index selects its zero-based row.
Existing output sequences are refused. Set POSE_CSV=None only to use automatic
CSV detection or, when no CSV exists, rounded filename labels.


In [ ]:
sequence = build_standard_gt(
    dataset=DATASET, output=OUTPUT, pose_csv=POSE_CSV,
    sequence_id=SEQUENCE_ID, map_name=MAP_NAME, carla_version=CARLA_VERSION,
)
print('Exported:', sequence)


In [ ]:
import json
import csv
import numpy as np
from geometry.poses import validate_poses
from datasets.standard_gt import StandardGTDataset

metadata = json.loads((sequence / 'metadata.json').read_text())
poses = np.load(sequence / 'poses_c2w.npy', allow_pickle=False)
with open(sequence / 'poses.csv', newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print('Frames:', len(rows))
print('Pose source:', metadata['pose_source'])
print('Matrix checks:', validate_poses(poses))
print('First CSV row:', rows[0])
print('First camera-to-world matrix:\n', poses[0])


In [ ]:
from IPython.display import display
from PIL import Image
# Preview only: the exported image itself is not resized.
with Image.open(sequence / rows[0]['image_name']) as image:
    preview = image.copy()
preview.thumbnail((1000, 500))
display(preview)

ds = StandardGTDataset(sequence)
print('Loader:', len(ds), 'frames; first RGB shape:', ds[0]['rgb'].shape)


## Next: model input inspection
From the project terminal:

```sh
python run_eval.py --standard-sequence Standard_GT/sequence001 --inspect-only --output outputs/standard-inspect01
```
For GPU inference, follow the README and supply `--checkpoint` instead of
`--inspect-only`. Pose evaluation metrics are not implemented yet.
These checks validate the conversion and file packaging; they do not independently
measure source image/pose alignment.
